# CockroachDB CDC - Load Parquet Files with Autoloader

This notebook demonstrates how to load CockroachDB CDC Parquet files using Databricks Autoloader.

## Prerequisites
- Azure Blob Storage with CDC Parquet files
- Databricks workspace with Unity Catalog enabled
- Configuration files: `.env/cockroachdb_cdc_azure.json` and `.env/cockroachdb_pipelines.json`


## Setup: Load Configuration and Define Variables

Load credentials from JSON files and define all necessary variables for the CDC pipeline.


In [ ]:
import json
import os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

print("="*80)
print("CONFIGURATION SETUP")
print("="*80)

# Find git root and construct paths to config files
# Assuming notebook is in sources/cockroachdb/notebooks/
#git_root = "/Workspace/Repos/lakeflow-community-connectors"  # Update this to your Databricks Repos path
git_root = os.path.abspath("../../..")
# Or use: git_root = os.path.abspath("../../..")

cockroach_dir = f"{git_root}/sources/cockroachdb"
azure_json_path = f"{cockroach_dir}/.env/cockroachdb_cdc_azure.json"
pipeline_json_path = f"{cockroach_dir}/.env/cockroachdb_pipelines.json"

# Alternative: Load from local filesystem if running locally
# azure_json_path = "../.env/cockroachdb_cdc_azure.json"
# pipeline_json_path = "../.env/cockroachdb_pipelines.json"

# Load Azure credentials
with open(azure_json_path, 'r') as f:
    azure_config = json.load(f)

# Load pipeline configuration
with open(pipeline_json_path, 'r') as f:
    pipeline_config = json.load(f)

# ============================================================================
# Azure Blob Storage Configuration
# ============================================================================
AZURE_STORAGE_ACCOUNT = azure_config["azure_storage_account"]
AZURE_STORAGE_KEY = azure_config["azure_storage_key"]
AZURE_CONTAINER = azure_config["azure_storage_container"]

# ============================================================================
# Source Path Configuration
# ============================================================================
# Azure Blob path format: wasbs://<container>@<account>.blob.core.windows.net/<prefix>/
PATH_PREFIX = pipeline_config["blob_prefix"]  # e.g., "parquet-cdc"
SOURCE_TABLE = "usertable"  # CockroachDB source table name

AZURE_SOURCE_PATH = f"wasbs://{AZURE_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net/{PATH_PREFIX}/"

# ============================================================================
# Target Configuration (Unity Catalog)
# ============================================================================
TARGET_CATALOG = pipeline_config["catalog"]       # e.g., "main"
TARGET_SCHEMA = pipeline_config["schema"]         # e.g., "robert_lee_cockroachdb"
TARGET_TABLE = f"{SOURCE_TABLE}_delta"            # Delta table name
TARGET_TABLE_PATH = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Unity Catalog Volume for data storage
VOLUME_NAME = pipeline_config["volume_name"]     # e.g., "parquet_files"
VOLUME_PATH = f"dbfs:/Volumes/{TARGET_CATALOG}/{TARGET_SCHEMA}/{VOLUME_NAME}"

# ============================================================================
# Autoloader Configuration (Using Unity Catalog Volume)
# ============================================================================
# Checkpoint location for Autoloader (tracks processed files)
# Store in Unity Catalog Volume instead of /tmp/ for durability
CHECKPOINT_PATH = f"{VOLUME_PATH}/_checkpoints/{SOURCE_TABLE}"

# Schema location (optional - for schema inference and evolution)
SCHEMA_LOCATION = f"{VOLUME_PATH}/_schemas/{SOURCE_TABLE}"

# ============================================================================
# CDC Configuration
# ============================================================================
# Primary key columns for deduplication and merge logic
PRIMARY_KEY_COLUMNS = ["ycsb_key"]  # Update based on your table schema

# ============================================================================
# Display Configuration
# ============================================================================
print("\n📋 Azure Storage Configuration:")
print(f"  Account: {AZURE_STORAGE_ACCOUNT}")
print(f"  Container: {AZURE_CONTAINER}")
print(f"  Prefix: {PATH_PREFIX}")
print(f"  Storage Key: {AZURE_STORAGE_KEY[:10]}... (loaded)")

print("\n📂 Source Path:")
print(f"  {AZURE_SOURCE_PATH}")

print("\n🎯 Target Configuration:")
print(f"  Catalog: {TARGET_CATALOG}")
print(f"  Schema: {TARGET_SCHEMA}")
print(f"  Table: {TARGET_TABLE}")
print(f"  Full Path: {TARGET_TABLE_PATH}")

print("\n📦 Unity Catalog Volume:")
print(f"  Volume: {VOLUME_NAME}")
print(f"  Path: {VOLUME_PATH}")

print("\n⚙️ Autoloader Configuration (Unity Catalog Volume):")
print(f"  Checkpoint: {CHECKPOINT_PATH}")
print(f"  Schema Location: {SCHEMA_LOCATION}")

print("\n🔑 CDC Configuration:")
print(f"  Primary Key: {PRIMARY_KEY_COLUMNS}")

print("\n✅ Configuration loaded successfully!")
print("="*80)


## Step 1: Load Parquet Files with Autoloader

Use Databricks Autoloader to incrementally load Parquet files from Azure Blob Storage.

**Key Features:**
- **Incremental Processing**: Only processes new files since last run
- **Schema Evolution**: Automatically handles schema changes
- **Fault Tolerance**: Checkpointing ensures exactly-once processing


In [ ]:
print("="*80)
print("STEP 1: LOAD PARQUET FILES WITH AUTOLOADER")
print("="*80)

# Read Parquet files using Autoloader (cloudFiles)
# Pass Azure credentials as Spark options for Databricks Spark Connect compatibility
df_raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useNotifications", "false")  # Use directory listing (simpler)
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_PATH}/schema")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("pathGlobFilter", f"*{SOURCE_TABLE}*.parquet")  # Filter for specific table
    # Pass Azure storage account key as option (works with Databricks Spark Connect)
    .option(f"fs.azure.account.key.{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net", AZURE_STORAGE_KEY)
    .load(AZURE_SOURCE_PATH)
)

print("✅ Autoloader stream configured")
print(f"📁 Source: {AZURE_SOURCE_PATH}")
print(f"📋 Schema location: {CHECKPOINT_PATH}/schema")
print(f"🔍 Filter: *{SOURCE_TABLE}*.parquet")
print(f"🔐 Authentication: Account key passed as stream option")
print("\n💡 Schema will be inferred from Parquet files")


## Step 2: Transform and Enrich Data

Add CDC metadata and prepare data for merge operations.

**CockroachDB Parquet CDC Fields:**
- `__crdb__event_type`: Event type ('c' = snapshot/update, 'i' = insert, 'd' = delete)
- `__crdb__updated`: Logical timestamp from CockroachDB

**Enhanced Fields Added:**
- `_cdc_operation`: Mapped operation (SNAPSHOT, INSERT, UPDATE, DELETE)
- `_cdc_timestamp`: Converted timestamp for easier querying
- `_source_file`: Originating file path for debugging
- `_processing_time`: When the record was processed by Autoloader


In [ ]:
print("="*80)
print("STEP 2: TRANSFORM AND ENRICH CDC DATA")
print("="*80)

# Add CDC metadata fields
df_enriched = (df_raw
    # Map __crdb__event_type to operation name
    # NOTE: In Parquet format, 'c' is used for SNAPSHOT, INSERT, AND UPDATE!
    # Cannot distinguish between these without additional logic (timestamp analysis)
    .withColumn("_cdc_operation",
        F.when(F.col("__crdb__event_type") == "c", F.lit("UPSERT"))  # c = snapshot/insert/update (indistinguishable)
         .when(F.col("__crdb__event_type") == "d", F.lit("DELETE"))
         .otherwise(F.lit("UNKNOWN"))
    )
    # Convert CockroachDB logical timestamp to readable format
    .withColumn("_cdc_timestamp", 
        F.col("__crdb__updated").cast("string")
    )
    # Add source file for debugging (Unity Catalog uses _metadata.file_path)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    
    # Add processing timestamp
    .withColumn("_processing_time", F.current_timestamp())
)

print("✅ Transformations applied:")
print("   - Mapped __crdb__event_type to _cdc_operation:")
print("     • 'c' → UPSERT (covers snapshot/insert/update - cannot distinguish)")
print("     • 'd' → DELETE (explicitly tracked)")
print("   - Converted __crdb__updated to _cdc_timestamp")
print("   - Added _source_file from _metadata.file_path (Unity Catalog)")
print("   - Added _processing_time")
print("\n⚠️  IMPORTANT: Parquet format uses 'c' for SNAPSHOT, INSERT, AND UPDATE")
print("   To distinguish:")
print("   - Use filename sequence (00000000 = snapshot, 00000001+ = CDC)")
print("   - Or use timestamp analysis (see PARQUET_UPDATE_DETECTION.md)")
print("   - For now, treating all 'c' events as UPSERT for Delta merge logic")


## Step 2b: Merge Column Family Fragments

**CRITICAL FIX**: When `split_column_families=true`, CockroachDB writes one Parquet file per column family.

**Problem:** 
- 9,995 keys × 11 column families = 109,945 fragment records ❌

**Solution:**
- Group by `ycsb_key` and merge all column families → 9,995 complete rows ✅


In [ ]:
import sys
import os
import importlib

# Add parent directory to path
parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import and reload to ensure latest version
import cockroachdb
importlib.reload(cockroachdb)

from cockroachdb import merge_column_family_fragments

print("="*80)
print("STEP 2B: MERGE COLUMN FAMILY FRAGMENTS")
print("="*80)

# Merge column family fragments
# For streaming DataFrames (from Autoloader), this always applies the merge
# For batch DataFrames, it auto-detects and only merges if needed
df_merged = merge_column_family_fragments(
    df_enriched,
    primary_key_columns=PRIMARY_KEY_COLUMNS,
    debug=True
)

# Replace df_enriched for downstream cells
df_enriched = df_merged

print("\n✅ Column family handling complete!")
print("="*80)


## Step 3: Write to Delta Table

Write the enriched CDC data to a Delta table using Autoloader's streaming capabilities.

**Write Mode**: `append` - Add all records to the Delta table
- Snapshot events are appended
- CDC events (INSERT, UPDATE, DELETE) are appended
- Custom merge logic can be applied in subsequent steps

**Checkpoint**: Ensures exactly-once processing and allows resumption after failures


In [ ]:
print("="*80)
print("STEP 3: WRITE TO DELTA TABLE")
print("="*80)

# Create target schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
print(f"✅ Schema ensured: {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Write stream to Delta table
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")  # Append all CDC events
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/delta")
    .option("mergeSchema", "true")  # Allow schema evolution
    .trigger(availableNow=True)  # Process all available files, then stop
    .toTable(TARGET_TABLE_PATH)
)

print(f"🚀 Stream started: {query.name}")
print(f"📊 Target: {TARGET_TABLE_PATH}")
print(f"📍 Checkpoint: {CHECKPOINT_PATH}/delta")
print("\n⏳ Processing files... (this may take a few moments)")

# Wait for stream to complete (trigger=availableNow means it will stop after processing)
query.awaitTermination()

print("\n✅ Stream completed!")

In [ ]:
print("="*80)
print("STEP 3: WRITE TO DELTA TABLE")
print("="*80)

# Create target schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_CATALOG}.{TARGET_SCHEMA}")
print(f"✅ Schema ensured: {TARGET_CATALOG}.{TARGET_SCHEMA}")

# Write stream to Delta table
query = (df_enriched.writeStream
    .format("delta")
    .outputMode("append")  # Append all CDC events
    .option("checkpointLocation", f"{CHECKPOINT_PATH}/delta")
    .option("mergeSchema", "true")  # Allow schema evolution
    .trigger(availableNow=True)  # Process all available files, then stop
    .toTable(TARGET_TABLE_PATH)
)

print(f"🚀 Stream started: {query.name}")
print(f"📊 Target: {TARGET_TABLE_PATH}")
print(f"📍 Checkpoint: {CHECKPOINT_PATH}/delta")
print("\n⏳ Processing files... (this may take a few moments)")

# Wait for stream to complete (trigger=availableNow means it will stop after processing)
query.awaitTermination()

print("\n✅ Stream completed!")


## Step 4: Verify Data Load

Check the loaded data and CDC statistics.


In [ ]:
print("="*80)
print("STEP 4: VERIFY DATA LOAD")
print("="*80)

# Read Delta table
df_delta = spark.table(TARGET_TABLE_PATH)

# Total record count
total_count = df_delta.count()
print(f"\n📊 Total records in Delta table: {total_count:,}")

# CDC operation breakdown
print(f"\n📊 CDC Operation Breakdown:")
cdc_stats = df_delta.groupBy("_cdc_operation").count().orderBy("count", ascending=False)
display(cdc_stats)

# Show sample records
print(f"\n📋 Sample Records (first 10):")
display(df_delta.limit(10))

# Table details
print(f"\n📊 Delta Table Details:")
display(spark.sql(f"DESCRIBE EXTENDED {TARGET_TABLE_PATH}").filter(
    F.col("col_name").isin(["Location", "Created Time", "Last Access", "Provider", "Table Properties"])
))


## Step 5: Verify Parquet Files Directly

Analyze the raw Parquet files from Azure Blob Storage to verify the CDC operation counts.

This uses the `analyze_azure_changefeed_files` utility function from `cockroachdb.py` to:
- Read all Parquet files from Azure
- Analyze __crdb__event_type values
- Deduplicate split_column_families fragments
- Count actual CDC operations

**Purpose:** Verify that the Delta table counts match the raw Parquet file counts.

In [ ]:
import sys
import os
import importlib

print("="*80)
print("STEP 5: VERIFY RAW PARQUET FILES FROM VOLUME")
print("="*80)

# Add parent directory to path to import cockroachdb module
parent_dir = os.path.abspath("../..")
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Import and reload cockroachdb module
import cockroachdb
importlib.reload(cockroachdb)

# Import utility function from cockroachdb module
from cockroachdb import analyze_volume_changefeed_files

print("\n🔍 Analyzing Parquet files from Unity Catalog Volume...")
print(f"   Volume: {VOLUME_PATH}")

try:
    # Analyze all Parquet files in the Volume
    # (with automatic deduplication for split_column_families)
    raw_stats = analyze_volume_changefeed_files(
        volume_path=VOLUME_PATH,
        debug=False  # Set to True for detailed debugging
    )
    
    print("\n✅ Volume File Analysis Complete!")
    print("="*80)
    
    # Display results
    print(f"\n📊 Unity Catalog Volume Statistics:")
    print(f"   Files analyzed: {raw_stats['file_count']}")
    print(f"   Unique keys (after deduplication): {raw_stats.get('unique_keys', 'N/A')}")
    print()
    print(f"   Operation Breakdown:")
    print(f"   ├─ UPSERT (snapshot/insert/update): {raw_stats['snapshot']:,}")
    print(f"   └─ DELETE:                          {raw_stats['delete']:,}")
    print(f"      ──────────────────────────────────")
    print(f"      Total:                           {raw_stats['snapshot'] + raw_stats['delete']:,}")
    
    # Compare with Delta table
    print(f"\n📊 Comparison with Delta Table:")
    print(f"   Delta table records: {total_count:,}")
    print(f"   Volume Parquet records: {raw_stats['snapshot'] + raw_stats['delete']:,}")
    
    if total_count == raw_stats['snapshot'] + raw_stats['delete']:
        print(f"   ✅ MATCH! Counts are identical.")
        print(f"   ✅ All files from Volume were successfully loaded into Delta table.")
    else:
        diff = total_count - (raw_stats['snapshot'] + raw_stats['delete'])
        print(f"   ⚠️  MISMATCH! Difference: {diff:,}")
        print(f"       This could be expected if:")
        print(f"       - Autoloader is still processing files")
        print(f"       - New files arrived in Volume after Delta load")
        print(f"       - Deduplication logic differs")
    
    print("\n" + "="*80)
    print("💡 INSIGHT: Parquet Event Type Mapping")
    print("="*80)
    print("""
    CockroachDB Parquet format uses:
    - 'c' = UPSERT (snapshot/insert/update combined - indistinguishable)
    - 'd' = DELETE (explicitly tracked)
    
    The raw count shows all 'c' events as UPSERT because Parquet format
    cannot distinguish between snapshot, insert, and update events without
    additional context (filename sequence or timestamp analysis).
    
    For Delta Lake merge operations, this is correct:
    - UPSERT → INSERT if not exists, UPDATE if exists
    - DELETE → DELETE from table
    """)
    
    print("\n" + "="*80)
    print("📋 Data Flow Verification")
    print("="*80)
    print(f"""
    ✅ Step 1: CockroachDB → Azure Blob Storage (via changefeed)
    ✅ Step 2: Azure → Unity Catalog Volume (via sync_azure_to_volume.sh)
    ✅ Step 3: Volume → Delta Table (via this notebook)
    
    This analysis verifies Step 3:
    - Source: {VOLUME_PATH}
    - Target: {TARGET_TABLE_PATH}
    - Files in Volume: {raw_stats['file_count']}
    - Records loaded: {total_count:,}
    """)
    
except Exception as e:
    print(f"\n❌ Error analyzing Volume files: {e}")
    print(f"\n💡 Common issues:")
    print(f"   - Volume path not accessible")
    print(f"   - No Parquet files in Volume (run sync_azure_to_volume.sh)")
    print(f"   - Missing SELECT permission on Volume")
    print(f"   - Spark session not available")
    print(f"\n   To fix:")
    print(f"   1. Ensure files are synced: ./sync_azure_to_volume.sh")
    print(f"   2. Grant permission: GRANT SELECT ON VOLUME ... TO `your.email`;")

## Summary

**✅ Successfully loaded Parquet CDC files from Azure Blob Storage into Databricks Delta tables!**

### Next Steps

1. **Query the Delta table:**
   ```sql
   SELECT * FROM {TARGET_TABLE_PATH}
   ORDER BY _cdc_timestamp DESC
   LIMIT 100
   ```

2. **Set up incremental pipeline:**
   - Schedule this notebook to run periodically
   - Autoloader will process only new files
   - Checkpoint ensures exactly-once processing

3. **Apply custom CDC merge logic:**
   - Use the connector's built-in CDC merge (recommended)
   - Or implement custom Delta MERGE operations
   - Handle column families if using `split_column_families`

4. **Monitor changefeed:**
   ```bash
   cd sources/cockroachdb/scripts
   ./test_azure_cdc.sh parquet stats
   ```

### Important Notes

**⚠️ Parquet Format:**
- Event type 'c' is used for BOTH snapshots and updates
- Use timestamp-based detection to distinguish (see PARQUET_UPDATE_DETECTION.md)
- For explicit before/after fields, use JSON format

**📚 Documentation:**
- [CDC_TEST_MATRIX_RESULTS.md](../learnings/CDC_TEST_MATRIX_RESULTS.md) - Comprehensive guide
- [PARQUET_UPDATE_DETECTION.md](../learnings/PARQUET_UPDATE_DETECTION.md) - UPDATE detection
- [TEST_AZURE_CDC_USAGE.md](../learnings/TEST_AZURE_CDC_USAGE.md) - Testing scripts


## Troubleshooting

### About the `wasbs://` URL format
**Question**: Why does the URL say "wasbs"? Are we using Wasabi storage?

**Answer**: No! `wasbs://` is Microsoft's naming convention for Azure Blob Storage:
- **WASBS** = **W**indows **A**zure **S**torage **B**lob **S**ecure (HTTPS)
- **WASB** = Windows Azure Storage Blob (HTTP)
- This is NOT related to Wasabi cloud storage

**Alternative**: You can also use `abfss://` (Azure Blob File System Secure) if you're using Azure Data Lake Storage Gen2:
```python
# For ADLS Gen2:
AZURE_SOURCE_PATH = f"abfss://{AZURE_CONTAINER}@{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net/{PATH_PREFIX}/"
```

### No files found in Azure
**Solution**: Run changefeed creation script:
```bash
cd sources/cockroachdb/scripts
./test_azure_cdc.sh parquet manual --force-new
```

### Authentication errors

**Error**: `CONFIG_NOT_AVAILABLE` when setting `spark.conf.set()`

**Explanation**: Databricks Spark Connect doesn't allow setting Azure storage configuration via `spark.conf.set()`.

**Solution**: This notebook now passes credentials as stream options instead:
```python
.option(f"fs.azure.account.key.{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net", AZURE_STORAGE_KEY)
```

**Alternative Solutions**:
1. **Databricks Secrets** (recommended for production):
   ```python
   AZURE_STORAGE_KEY = dbutils.secrets.get(scope="azure-storage", key="storage-key")
   ```

2. **Unity Catalog External Locations** (most secure):
   - Create external location pointing to your Azure container
   - Grant permissions to your user/service principal
   - Authentication is automatic

**Verify credentials**:
- Check `azure_storage_account` name in `cockroachdb_cdc_azure.json`
- Check `azure_storage_key` is valid (not expired)
- Check `azure_storage_container` exists

### Schema evolution errors
**Solution**: Delete checkpoint and restart:
```python
dbutils.fs.rm(CHECKPOINT_PATH, True)
```

### Parquet UPDATE detection
**Issue**: All events appear as SNAPSHOT (type 'c')

**Explanation**: This is expected behavior. The Parquet format uses 'c' for both snapshots and updates.

**Solution**: 
- Use the connector's built-in timestamp-based detection
- Or use JSON format for explicit before/after fields
- **Reference**: See `PARQUET_UPDATE_DETECTION.md` for details

### Column families
**Issue**: Multiple Parquet files per table

**Explanation**: Tables with `split_column_families` create separate files per column family.

**Solution**: The Autoloader pattern handles this automatically - all files for the table are merged.

**Reference**: See `CDC_TEST_MATRIX_RESULTS.md` for column family behavior
